# Model Selection in Python (scikit-learn)
Simple guide with examples.

**Command line:**
```
pip install numpy scikit-learn jupyter
jupyter notebook
```
Model selection = how we **split** data and **test** a model, so it works on new data.

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import (
    train_test_split,
    KFold, StratifiedKFold, RepeatedKFold,
    ShuffleSplit, StratifiedShuffleSplit,
    LeaveOneOut, LeavePOut,
    GroupKFold, LeaveOneGroupOut,
    TimeSeriesSplit,
    cross_val_score, cross_validate, cross_val_predict,
    GridSearchCV, RandomizedSearchCV,
    learning_curve, validation_curve,
)

# ---------------------------------------------------------
# SAMPLE DATA (made with numpy)
# X = features (inputs), y = target (answer we want to predict)
# ---------------------------------------------------------
np.random.seed(42)
X = np.random.rand(20, 3)                 # 20 rows, 3 columns
y = np.array([0, 1] * 10)                 # 0 and 1 (10 each)
groups = np.repeat([1, 2, 3, 4], 5)       # 4 groups (example: 4 customers)

model = LogisticRegression()

## Part 1: Splitting data

### 1) train_test_split  (MOST USED)
Splits data into a training part and a testing part.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,       # 20% for testing
    random_state=42,     # same split every time
    stratify=y,          # keep class ratio the same
)
print("Train rows:", X_train.shape[0], "| Test rows:", X_test.shape[0])

### 2) KFold
Cuts data into K parts. Each part is the test set one time.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
for i, (train_idx, test_idx) in enumerate(kf.split(X), start=1):
    print(f"Fold {i}: train={len(train_idx)}, test={len(test_idx)}")

### 3) StratifiedKFold
Like KFold, but keeps the class ratio in every fold.
Use for CLASSIFICATION, especially when classes are not balanced.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for i, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
    print(f"Fold {i}: test class counts = {np.bincount(y[test_idx])}")

### 4) RepeatedKFold
Runs KFold many times with different shuffles. More reliable score.

In [ ]:
rkf = RepeatedKFold(n_splits=5, n_repeats=2, random_state=42)
print("Total splits:", rkf.get_n_splits())

### 5) ShuffleSplit
Random train/test splits, repeated. Test sets can overlap.

In [ ]:
ss = ShuffleSplit(n_splits=3, test_size=0.25, random_state=42)
for i, (train_idx, test_idx) in enumerate(ss.split(X), start=1):
    print(f"Split {i}: train={len(train_idx)}, test={len(test_idx)}")

### 6) StratifiedShuffleSplit
ShuffleSplit + keeps the class ratio.

In [ ]:
sss = StratifiedShuffleSplit(n_splits=3, test_size=0.25, random_state=42)
for i, (train_idx, test_idx) in enumerate(sss.split(X, y), start=1):
    print(f"Split {i}: test class counts = {np.bincount(y[test_idx])}")

### 7) LeaveOneOut (LOO)
Test set = 1 row. Repeat for every row. Good for VERY small data.
Slow on big data!

In [ ]:
loo = LeaveOneOut()
print("Number of splits:", loo.get_n_splits(X))

### 8) LeavePOut
Test set = P rows, all possible combinations. Very slow. Rarely used.

In [ ]:
lpo = LeavePOut(p=2)
print("Number of splits:", lpo.get_n_splits(X[:6]), "(using only 6 rows)")

### 9) GroupKFold
Rows from the SAME group never go to both train and test.
Example: same customer must not appear in both.

In [ ]:
gkf = GroupKFold(n_splits=4)
for i, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups), start=1):
    print(f"Fold {i}: test groups = {np.unique(groups[test_idx])}")

### 10) LeaveOneGroupOut
Each group becomes the test set one time.

In [ ]:
logo = LeaveOneGroupOut()
for i, (train_idx, test_idx) in enumerate(logo.split(X, y, groups), start=1):
    print(f"Split {i}: test group = {np.unique(groups[test_idx])}")

### 11) TimeSeriesSplit
For data with TIME order (sales by date). Train = past, Test = future.
Never shuffle time data!

In [ ]:
tss = TimeSeriesSplit(n_splits=4)
for i, (train_idx, test_idx) in enumerate(tss.split(X), start=1):
    print(f"Split {i}: train rows 0-{train_idx[-1]}, "
          f"test rows {test_idx[0]}-{test_idx[-1]}")

## Part 2: Checking the model (cross validation)

### 12) cross_val_score
Easiest way: gives one score per fold.

In [ ]:
scores = cross_val_score(model, X, y, cv=5, scoring="accuracy")
print("Scores:", np.round(scores, 2))
print("Average:", round(scores.mean(), 2))

### 13) cross_validate
Like cross_val_score, but can give many scores + fit time.

In [ ]:
result = cross_validate(model, X, y, cv=5,
                        scoring=["accuracy", "f1"],
                        return_train_score=True)
print("Test accuracy:", np.round(result["test_accuracy"], 2))
print("Test f1      :", np.round(result["test_f1"], 2))

### 14) cross_val_predict
Gives the prediction for every row (made when that row was in test).

In [ ]:
preds = cross_val_predict(model, X, y, cv=5)
print("Predictions:", preds)

## Part 3: Finding the best settings

### 15) GridSearchCV
Tries ALL combinations of settings. Picks the best.

In [ ]:
param_grid = {"max_depth": [1, 2, 3, 4], "min_samples_split": [2, 4]}
grid = GridSearchCV(DecisionTreeClassifier(random_state=42),
                    param_grid, cv=3, scoring="accuracy")
grid.fit(X, y)
print("Best settings:", grid.best_params_)
print("Best score   :", round(grid.best_score_, 2))

### 16) RandomizedSearchCV
Tries only RANDOM combinations. Faster when there are many settings.

In [ ]:
param_dist = {"max_depth": list(range(1, 10)),
              "min_samples_split": list(range(2, 10))}
rand = RandomizedSearchCV(DecisionTreeClassifier(random_state=42),
                          param_dist, n_iter=5, cv=3, random_state=42)
rand.fit(X, y)
print("Best settings:", rand.best_params_)

## Part 4: Overfitting / underfitting check

### 17) learning_curve
Score when we use more and more training data.

In [ ]:
sizes, train_sc, test_sc = learning_curve(
    model, X, y, cv=3, train_sizes=[0.4, 0.7, 1.0])
print("Train sizes :", sizes)
print("Train score :", np.round(train_sc.mean(axis=1), 2))
print("Test score  :", np.round(test_sc.mean(axis=1), 2))

### 18) validation_curve
Score when we change ONE setting.

In [ ]:
depths = np.array([1, 2, 3, 4, 5])
train_sc, test_sc = validation_curve(
    DecisionTreeClassifier(random_state=42), X, y,
    param_name="max_depth", param_range=depths, cv=3)
print("max_depth  :", depths)
print("Train score:", np.round(train_sc.mean(axis=1), 2))
print("Test score :", np.round(test_sc.mean(axis=1), 2))

## Quick summary
| Need | Use |
|---|---|
| Simple split | `train_test_split` |
| Normal cross validation | `KFold` |
| Classification | `StratifiedKFold` |
| Very small data | `LeaveOneOut` |
| Same customer/group | `GroupKFold`, `LeaveOneGroupOut` |
| Time / date data | `TimeSeriesSplit` |
| Quick score | `cross_val_score` |
| Many scores | `cross_validate` |
| Best settings | `GridSearchCV`, `RandomizedSearchCV` |
| Overfitting check | `learning_curve`, `validation_curve` |